In [1]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import json
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("../shared").resolve()))
sys.path.insert(0, str(Path(".").resolve()))
sys.path.insert(0, str(Path("../..").resolve()))

from constants import (
    CAPTIONS_PATH,
    FIGURES_PATH,
)
from pgf_utils import (
    apply_figure_style,
    configure_pgf,
    configure_screen,
    figure_inches,
    save_caption,
    save_pgf,
)
from submission import (
    ARM_LINK,
    CARTESIAN_CONTROL_CACHE_PATH,
    CC_ABLATION_RESULTS_DIR,
    CC_AMP_REG,
    CC_A_MAX_EL_NEG,
    CC_A_MAX_EL_POS,
    CC_A_MAX_SH_NEG,
    CC_A_MAX_SH_POS,
    CC_BAND_CHANNELS,
    CC_CONSTRAIN_ELBOW,
    CC_DIST_THRESH,
    CC_EL_THRESH,
    CC_N_R,
    CC_N_THETA,
    CC_OPEN_LOOP_OFFSET,
    CC_OPTIMAL_PARAMS_PATH,
    CC_Q_INT,
    CC_Q_TRACK,
    CC_R_EFFORT,
    CC_SH_DRIFT_THRESH,
    CC_SH_THRESH,
    CC_T,
    CC_TARGET_R_MAX,
    CC_TARGET_R_MIN,
    CC_USE_LQI,
    ERA_EM_4_PATH,
    MAX_DELTA,
    aggregate_seed_metrics,
    run_cartesian_control,
)
from cartesian_control_ablation_blend import (
    NOTEBOOK_GITHUB_URL,
    make_blend_figure,
)

In [2]:
# ── 2. Constants ──────────────────────────────────────────────────────────────
ABLATION_SEEDS = list(range(10))
BLEND_ALPHAS = [0.05, 0.10, 0.15, 0.25, 0.40]

In [3]:
# ── 3. Derived variables ──────────────────────────────────────────────────────
if CC_OPTIMAL_PARAMS_PATH.exists():
    with open(CC_OPTIMAL_PARAMS_PATH) as fh:
        optimal_params = json.load(fh)
    print(f"Loaded optimal params from {CC_OPTIMAL_PARAMS_PATH.name}")
else:
    optimal_params = {}
    print("Optimal params not found — using defaults from constants.py")

CC_ABLATION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

base_params = dict(
    cache_path=CARTESIAN_CONTROL_CACHE_PATH,
    estimator_path=ERA_EM_4_PATH,
    T=CC_T,
    n_r=CC_N_R,
    n_theta=CC_N_THETA,
    target_r_min=CC_TARGET_R_MIN,
    target_r_max=CC_TARGET_R_MAX,
    dist_thresh=CC_DIST_THRESH,
    arm_link=ARM_LINK,
    max_delta=MAX_DELTA,
    q_track=CC_Q_TRACK,
    r_effort=CC_R_EFFORT,
    q_int=CC_Q_INT,
    el_thresh=CC_EL_THRESH,
    sh_thresh=CC_SH_THRESH,
    sh_drift_thresh=CC_SH_DRIFT_THRESH,
    a_max_sh_pos=CC_A_MAX_SH_POS,
    a_max_sh_neg=CC_A_MAX_SH_NEG,
    a_max_el_pos=CC_A_MAX_EL_POS,
    a_max_el_neg=CC_A_MAX_EL_NEG,
    amp_reg=CC_AMP_REG,
    constrain_elbow=CC_CONSTRAIN_ELBOW,
    use_lqi=CC_USE_LQI,
    band_channels=CC_BAND_CHANNELS,
    open_loop_offset=CC_OPEN_LOOP_OFFSET,
)

Loaded optimal params from cc_optimal_params.json


In [4]:
# ── 4. Simulation ─────────────────────────────────────────────────────────────
ablation_results = {}
for alpha in BLEND_ALPHAS:
    per_seed: list = []
    for seed in ABLATION_SEEDS:
        slug = f"{alpha:.2f}".replace(".", "p")
        _call_kwargs: dict[str, Any] = {
            **base_params,
            **optimal_params,
            "seq_blend_alpha": alpha,
        }
        results, _, _ = run_cartesian_control(
            **_call_kwargs,
            demo_seed=seed,
            results_path=CC_ABLATION_RESULTS_DIR / f"blend_{slug}_{seed}.pkl",
        )
        per_seed.append(results)
    ablation_results[alpha] = aggregate_seed_metrics(
        per_seed, T=CC_T, dist_thresh=CC_DIST_THRESH
    )

print("\nAblation summary (mean ± SD across seeds):")
for alpha, agg in ablation_results.items():
    fd = agg["final_dist"]
    fr = agg["frac_in_sector"]
    print(
        f"  alpha={alpha:.2f}: final_dist={fd.mean():.2f}±{fd.std():.2f}  frac={fr.mean():.3f}±{fr.std():.3f}"
    )

Loading cached results from blend_0p05_0.pkl
Loading cached results from blend_0p05_1.pkl
Loading cached results from blend_0p05_2.pkl
Loading cached results from blend_0p05_3.pkl
Loading cached results from blend_0p05_4.pkl
Loading cached results from blend_0p05_5.pkl
Loading cached results from blend_0p05_6.pkl
Loading cached results from blend_0p05_7.pkl
Loading cached results from blend_0p05_8.pkl
Loading cached results from blend_0p05_9.pkl
Loading cached results from blend_0p10_0.pkl
Loading cached results from blend_0p10_1.pkl
Loading cached results from blend_0p10_2.pkl
Loading cached results from blend_0p10_3.pkl
Loading cached results from blend_0p10_4.pkl
Loading cached results from blend_0p10_5.pkl
Loading cached results from blend_0p10_6.pkl
Loading cached results from blend_0p10_7.pkl
Loading cached results from blend_0p10_8.pkl
Loading cached results from blend_0p10_9.pkl
Loading cached results from blend_0p15_0.pkl
Loading cached results from blend_0p15_1.pkl
Loading ca

In [5]:
# ── 5. Display ────────────────────────────────────────────────────────────────
configure_screen()
fig = make_blend_figure(ablation_results, figsize=(10.0, 2.5))
apply_figure_style(fig)
plt.show()

ValueError: ANOVA (alpha=0.05) found no significant metrics. The ablation parameter may have no effect, or there is insufficient variance across seeds.

In [ ]:
# ── 6. PGF export ─────────────────────────────────────────────────────────────
FIG_WIDTH_FRAC = 0.50
FIG_HEIGHT_FRAC = 0.14

configure_pgf()
fig = make_blend_figure(
    ablation_results,
    figsize=figure_inches(FIG_WIDTH_FRAC, FIG_HEIGHT_FRAC),
)
save_pgf(fig, FIGURES_PATH / "cartesian_control_ablation_blend.pgf")
plt.close(fig)

In [ ]:
# ── 7. Caption ────────────────────────────────────────────────────────────────
alpha_vals_str = ", ".join(str(a) for a in BLEND_ALPHAS)

caption = (
    r"Ablation over the idle-joint blend parameter $\alpha_\mathrm{blend} \in \{"
    + alpha_vals_str
    + r"\}$; setup and encoding as "
    r"\autoref{fig:cartesian_control_ablation_band_scale}."
)

save_caption(
    CAPTIONS_PATH / "cartesian_control_ablation_blend.tex",
    caption,
    NOTEBOOK_GITHUB_URL,
)
print(caption)